# WavLM Embedding Extraction

In [ ]:
# Setup
!pip install transformers datasets librosa accelerate torch kaggle -q
import os, json, numpy as np, torch
import librosa
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from tqdm.auto import tqdm
print('Setup done', flush=True)


In [ ]:
# Kaggle credentials
!mkdir -p /root/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"subhajitdas","key":"YOUR_KAGGLE_KEY"}')
print('Credentials set', flush=True)


In [ ]:
# Download dataset
!kaggle datasets download -d subhajitdas/chuckle-net-prosody-fusion -p /content/chuckle-net --force
!cd /content/chuckle-net && unzip -o chuckle-net-prosody-fusion.zip
AUDIO_DIR = '/content/chuckle-net/audio'
ALIGNED_PATH = '/content/chuckle-net/aligned_utterances.jsonl'
print('Audio files:', len(os.listdir(AUDIO_DIR)), flush=True)


In [ ]:
# Load WavLM
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to(device)
wavlm.eval()
fext = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
SR = 16000
DUR = 5.0
MAX_LEN = int(DUR * SR)
print('WavLM loaded', flush=True)


In [ ]:
# Load data
aligned_data = [json.loads(l) for l in open(ALIGNED_PATH)]
print('Total:', len(aligned_data), flush=True)
utt_meta = {}
for d in aligned_data:
    uid = d['video_id'] + '_' + str(d['start'])
    utt_meta[uid] = {'video_id': d['video_id'], 'start': d['start'], 'end': d['end'], 'label': d.get('label', 0)}
print('Unique:', len(utt_meta), flush=True)


In [ ]:
# Extract embeddings
def extract_emb(audio_path, start):
    try:
        off = max(0, start - 0.05)
        audio, _ = librosa.load(audio_path, sr=SR, mono=True, offset=off, duration=DUR)
        if len(audio) < MAX_LEN:
            audio = np.pad(audio, (0, MAX_LEN - len(audio)))
        audio = audio[:MAX_LEN]
        inputs = fext([audio], sampling_rate=SR, return_tensors='pt', padding=False)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            hidden = wavlm(**inputs).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return hidden.tolist()
    except:
        return [0.0] * 768

embeddings = {}
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.mp3')]
for af in tqdm(audio_files):
    vid = af.replace('.mp3', '')
    path = os.path.join(AUDIO_DIR, af)
    for uid, u in utt_meta.items():
        if u['video_id'] == vid:
            embeddings[uid] = {
                'emb': extract_emb(path, u['start']),
                'label': u['label']
            }
    if len(embeddings) % 1000 == 0:
        print('Done:', len(embeddings), flush=True)

print('Total embeddings:', len(embeddings), flush=True)


In [ ]:
# Save
OUTPUT = '/content/chuckle-net/wavlm_embeddings.json'
with open(OUTPUT, 'w') as f:
    json.dump(embeddings, f)
print('Saved to', OUTPUT, flush=True)
pos = sum(v['label'] for v in embeddings.values())
print('Positive:', pos, 'Negative:', len(embeddings) - pos, flush=True)
print('DONE!', flush=True)
